## Experiment Setup
Configure tested models and judge sources for consensus analysis across UX evaluation dimensions.

In [31]:
from pathlib import Path
import json
import warnings
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd

# Configurable paths and names
BASE_SCORE_DIR = "<PROJECT_ROOT>/Desktop/Desktop - ADUAED19365LPMX/Agent_IX_Personalization/gorilla/berkeley-function-call-leaderboard/LLM_as_judge_score"
TESTED_MODELS = [
    "claude_opus_4_5_20251101_FC",
    "claude_sonnet_4_5_20250929_FC",
    "gemini_3_flash_FC",
    "kimi_k2_0905_preview_FC",
]

JUDGE_NAMES = [
    "anthropic_claude-opus-4.5",
    "anthropic_claude-sonnet-4.5",
    "google_gemini-3-flash-preview",
    "moonshotai_kimi-k2-0905",
]

CONDITIONS = ["no_personalization", "personalization"]

# Persona enumeration (fixed set used during evaluation)
PERSONAS = [
    "chain_parallel",
    "chain_sequential",
    "confirmation_batch",
    "disambiguation_gradual",
    "disambiguation_upfront",
    "each_confirmation",
    "error_discovery_brief",
    "error_discovery_detail",
    "error_retry_escalation",
    "error_retry_silent",
    "info_collect_gradual",
    "info_collect_upfront",
    "param_high",
    "param_low",
    "param_medium",
    "presentation_compact",
    "presentation_layered",
    "silent",
    "source_high",
    "source_low",
    "tool_abortion_continue",
    "tool_abortion_stop",
    "tool_high",
    "tool_initiative_proactive",
    "tool_initiative_reactive",
    "tool_invocation_multiple",
    "tool_invocation_single",
    "tool_low",
    "tool_medium",
    "tool_switch_high_agency",
    "tool_switch_low_agency",
]

# Fixed UX evaluation dimensions emitted by judge models
DIMENSIONS: List[str] = [
    "initiative_timing",
    "interaction_coherence",
    "intent_alignment_drift",
    "commitment_consistency",
    "interaction_preference_alignment",
    "interaction_efficiency",
    "user_cognitive_load_trajectory",
    "overall_user_experience",
]

pd.set_option("display.max_rows", 20)
pd.set_option("display.max_columns", None)


## Data Loading
Parse judge outputs and normalize schema across models/personas.

In [32]:
def load_scores(directory: Path, persona: str | None = None) -> pd.DataFrame:
    files = sorted(directory.glob("*_judge.json"))
    if not files:
        raise FileNotFoundError(f"No *_judge.json files found in {directory}")

    rows = []
    for fp in files:
        try:
            content = json.loads(fp.read_text(encoding="utf-8"))
        except json.JSONDecodeError as exc:
            warnings.warn(f"Failed to parse JSON for {fp.name}: {exc}")
            continue

        parsed = content.get("parsed")
        if isinstance(parsed, dict) and parsed.get("dimensions"):
            dims_block = parsed.get("dimensions") or {}
        else:
            dims_block = content.get("dimensions") or {}
            if not dims_block:
                dim_name = content.get("dimension")
                score_val = content.get("score")
                if dim_name in DIMENSIONS and isinstance(score_val, (int, float)):
                    dims_block = {
                        dim_name: {
                            "score": float(score_val),
                            "justification": content.get("justification", ""),
                            "evidence_turn_ids": content.get("evidence_turn_ids", []),
                        }
                    }
                else:
                    dims_block = {}

        row = {"test_id": fp.stem.replace("_judge", "")}
        if persona:
            row["persona"] = persona

        for dim in DIMENSIONS:
            dim_entry = dims_block.get(dim)
            score = None if dim_entry is None else dim_entry.get("score")
            if isinstance(score, (int, float)):
                row[dim] = float(score)
            else:
                warnings.warn(f"Missing/invalid score for '{dim}' in {fp.name}; set NaN")
                row[dim] = np.nan
        rows.append(row)

    if not rows:
        raise ValueError(f"No valid records loaded from {directory}")

    columns = ["test_id"] + (["persona"] if persona else []) + DIMENSIONS
    return pd.DataFrame(rows, columns=columns)


## Aggregate All Models
Load every tested model across all judge models and conditions; store per-model dictionaries and a long-format DataFrame for later consensus computation.

In [33]:
def collect_all_scores(models: List[str], judges: List[str], personas: List[str], conditions: List[str]):
    aggregated: Dict[str, Dict[str, Dict[str, pd.DataFrame]]] = {}
    combined_frames: List[pd.DataFrame] = []
    empty_columns = ["tested_model", "judge_model", "condition", "test_id", "persona"] + DIMENSIONS

    for model in models:
        aggregated[model] = {}
        for judge in judges:
            aggregated[model][judge] = {}
            for condition in conditions:
                persona_frames: List[pd.DataFrame] = []
                for persona in personas:
                    dir_path = Path(BASE_SCORE_DIR) / judge / model / condition / persona
                    if not dir_path.exists() or not any(dir_path.iterdir()):
                        continue

                    try:
                        df = load_scores(dir_path, persona=persona)
                    except Exception as exc:
                        warnings.warn(f"Skip {dir_path}: {exc}")
                        continue

                    df.insert(0, "condition", condition)
                    df.insert(0, "judge_model", judge)
                    df.insert(0, "tested_model", model)
                    persona_frames.append(df)

                if persona_frames:
                    combined = pd.concat(persona_frames, ignore_index=True)
                    aggregated[model][judge][condition] = combined
                    combined_frames.append(combined)
                else:
                    aggregated[model][judge][condition] = pd.DataFrame(columns=empty_columns)

    all_scores_long = (
        pd.concat(combined_frames, ignore_index=True)
        if combined_frames
        else pd.DataFrame(columns=empty_columns)
    )
    return aggregated, all_scores_long


all_scores_by_model, all_scores_long = collect_all_scores(TESTED_MODELS, JUDGE_NAMES, PERSONAS, CONDITIONS)
print(f"Loaded {len(all_scores_long)} rows across {len(TESTED_MODELS)} tested models and {len(JUDGE_NAMES)} judge models.")

summary = (
    all_scores_long.groupby(["tested_model", "judge_model", "condition"])
    .agg(num_samples=("test_id", "count"))
    .reset_index()
    .sort_values(["tested_model", "judge_model", "condition"])
)
summary.head(20)


/var/folders/h0/hg82lwhs2l1bvq786x_jh8xh0000gs/T/ipykernel_3898/4062961482.py:43: UserWarning: Missing/invalid score for 'interaction_preference_alignment' in multi_turn_long_context_104_judge.json; set NaN
  warnings.warn(f"Missing/invalid score for '{dim}' in {fp.name}; set NaN")
/var/folders/h0/hg82lwhs2l1bvq786x_jh8xh0000gs/T/ipykernel_3898/4062961482.py:43: UserWarning: Missing/invalid score for 'intent_alignment_drift' in multi_turn_long_context_27_judge.json; set NaN
  warnings.warn(f"Missing/invalid score for '{dim}' in {fp.name}; set NaN")
/var/folders/h0/hg82lwhs2l1bvq786x_jh8xh0000gs/T/ipykernel_3898/4062961482.py:43: UserWarning: Missing/invalid score for 'user_cognitive_load_trajectory' in multi_turn_long_context_35_judge.json; set NaN
  warnings.warn(f"Missing/invalid score for '{dim}' in {fp.name}; set NaN")
/var/folders/h0/hg82lwhs2l1bvq786x_jh8xh0000gs/T/ipykernel_3898/4062961482.py:43: UserWarning: Missing/invalid score for 'user_cognitive_load_trajectory' in multi_tu

Loaded 8872 rows across 4 tested models and 4 judge models.


/var/folders/h0/hg82lwhs2l1bvq786x_jh8xh0000gs/T/ipykernel_3898/4062961482.py:43: UserWarning: Missing/invalid score for 'interaction_coherence' in multi_turn_miss_param_14_judge.json; set NaN
  warnings.warn(f"Missing/invalid score for '{dim}' in {fp.name}; set NaN")
/var/folders/h0/hg82lwhs2l1bvq786x_jh8xh0000gs/T/ipykernel_3898/4062961482.py:43: UserWarning: Missing/invalid score for 'user_cognitive_load_trajectory' in multi_turn_miss_param_14_judge.json; set NaN
  warnings.warn(f"Missing/invalid score for '{dim}' in {fp.name}; set NaN")
/var/folders/h0/hg82lwhs2l1bvq786x_jh8xh0000gs/T/ipykernel_3898/4062961482.py:43: UserWarning: Missing/invalid score for 'interaction_coherence' in multi_turn_miss_param_101_judge.json; set NaN
  warnings.warn(f"Missing/invalid score for '{dim}' in {fp.name}; set NaN")
/var/folders/h0/hg82lwhs2l1bvq786x_jh8xh0000gs/T/ipykernel_3898/4062961482.py:43: UserWarning: Missing/invalid score for 'user_cognitive_load_trajectory' in multi_turn_miss_param_14_j

,tested_model,judge_model,condition,num_samples
0,claude_opus_4_5_20251101_FC,anthropic_claude-opus-4.5,no_personalization,282
1,claude_opus_4_5_20251101_FC,anthropic_claude-opus-4.5,personalization,282
2,claude_opus_4_5_20251101_FC,anthropic_claude-sonnet-4.5,no_personalization,282
3,claude_opus_4_5_20251101_FC,anthropic_claude-sonnet-4.5,personalization,282
4,claude_opus_4_5_20251101_FC,google_gemini-3-flash-preview,no_personalization,282
5,claude_opus_4_5_20251101_FC,google_gemini-3-flash-preview,personalization,280
6,claude_opus_4_5_20251101_FC,moonshotai_kimi-k2-0905,no_personalization,282
7,claude_opus_4_5_20251101_FC,moonshotai_kimi-k2-0905,personalization,282
8,claude_sonnet_4_5_20250929_FC,anthropic_claude-opus-4.5,no_personalization,282
9,claude_sonnet_4_5_20250929_FC,anthropic_claude-opus-4.5,personalization,282


## Judge Consensus Metrics
Compute disagreement per tested model and dimension, then aggregate to global dimension-level consensus and ICC.

In [34]:
# import matplotlib.pyplot as plt
# import seaborn as sns

# def icc_two_way_random_absolute(scores: pd.DataFrame) -> float:
#     """
#     Computes ICC(2,k) according to Shrout & Fleiss (1979).
#     ICC(2,k) is two-way random, absolute agreement, average measures.
#     """
#     scores_clean = scores.dropna(axis=0, how="any")
#     n, k = scores_clean.shape
#     if n < 2 or k < 2:
#         return np.nan

#     mean_target = scores_clean.mean(axis=1)   # mean per sample
#     mean_rater = scores_clean.mean(axis=0)    # mean per judge
#     grand_mean = scores_clean.values.mean()

#     # Convert Series to numpy arrays for broadcasting.
#     mean_target_np = mean_target.to_numpy()[:, None]   # shape (n, 1)
#     mean_rater_np = mean_rater.to_numpy()[None, :]     # shape (1, k)
#     scores_np = scores_clean.to_numpy()                # shape (n, k)

#     residual = scores_np - mean_target_np - mean_rater_np + grand_mean
#     ss_error = (residual ** 2).sum()

#     ss_target = k * ((mean_target_np.flatten() - grand_mean) ** 2).sum()
#     ss_rater = n * ((mean_rater_np.flatten() - grand_mean) ** 2).sum()

#     ms_target = ss_target / (n - 1)
#     ms_rater = ss_rater / (k - 1)
#     ms_error = ss_error / ((n - 1) * (k - 1))

#     numerator = ms_target - ms_error
#     denominator = ms_target + (ms_rater - ms_error) / n
#     if denominator == 0:
#         return np.nan
#     return numerator / denominator


# # 1) Per tested_model & dimension: std across the 4 judges (averaged over samples)
# std_records = []
# for dim in DIMENSIONS:
#     pivot = (
#         all_scores_long[["tested_model", "condition", "persona", "test_id", "judge_model", dim]]
#         .pivot_table(
#             index=["tested_model", "condition", "persona", "test_id"],
#             columns="judge_model",
#             values=dim,
#         )
#     )
#     pivot = pivot.dropna(how="any")
#     if pivot.empty:
#         continue
#     sample_std = pivot.std(axis=1, ddof=1).rename("std_across_judges")
#     tmp = sample_std.reset_index()
#     tmp["dimension"] = dim
#     std_records.append(tmp)

# if std_records:
#     std_df = pd.concat(std_records, ignore_index=True)
# else:
#     std_df = pd.DataFrame(columns=["tested_model", "condition", "persona", "test_id", "std_across_judges", "dimension"])

# # Average disagreement per tested_model/dimension (collapse conditions/personas/tests)
# model_dim_disagreement = (
#     std_df.groupby(["tested_model", "dimension"])["std_across_judges"]
#     .mean()
#     .reset_index()
#     .rename(columns={"std_across_judges": "mean_std_across_judges"})
# )

# # 2) Global dimension-level consensus: average std across all tested models
# if not model_dim_disagreement.empty:
#     dim_disagreement = (
#         model_dim_disagreement.groupby("dimension")["mean_std_across_judges"].mean().reset_index()
#     )
# else:
#     dim_disagreement = pd.DataFrame(columns=["dimension", "mean_std_across_judges"])

# # 3) ICC per dimension (using all models/conditions/personas)
# icc_values = {}
# for dim in DIMENSIONS:
#     pivot = (
#         all_scores_long[["tested_model", "condition", "persona", "test_id", "judge_model", dim]]
#         .pivot_table(
#             index=["tested_model", "condition", "persona", "test_id"],
#             columns="judge_model",
#             values=dim,
#         )
#     )
#     pivot = pivot.dropna(how="any")
#     if pivot.shape[0] < 2:
#         icc_values[dim] = np.nan
#     else:
#         icc_values[dim] = icc_two_way_random_absolute(pivot)

# # 4) Dimension summary table: mean score, mean disagreement, ICC
# mean_scores = {dim: all_scores_long[dim].mean() for dim in DIMENSIONS}
# dim_summary = pd.DataFrame({
#     "dimension": DIMENSIONS,
#     "mean_score": [mean_scores.get(d, np.nan) for d in DIMENSIONS],
#     "mean_std": [dim_disagreement.set_index("dimension").get("mean_std_across_judges").get(d, np.nan) if not dim_disagreement.empty else np.nan for d in DIMENSIONS],
#     "icc": [icc_values.get(d, np.nan) for d in DIMENSIONS],
# })

# # Display key tables
# print("Per tested model + dimension disagreement (first 10 rows):")
# print(model_dim_disagreement.head(10))
# print("Dimension-level consensus (mean std across models):")
# print(dim_disagreement)
# print("ICC per dimension:")
# print(pd.Series(icc_values))
# dim_summary


## ICC (unique target per model/persona)
Recompute ICC for each dimension using a unique target key (`tested_model|condition|persona|test_id`) so pingouin does not merge identical `test_id` across models/personas.

In [35]:
icc_unique = {}
for dim in DIMENSIONS:
    scores_long = all_scores_long[["tested_model", "condition", "persona", "test_id", "judge_model", dim]].copy()
    scores_long = scores_long.rename(columns={dim: "scores"})
    scores_long = scores_long.dropna(subset=["scores"])
    scores_long["unique_target"] = (
        scores_long["tested_model"]
        + "|" + scores_long["condition"]
        + "|" + scores_long["persona"]
        + "|" + scores_long["test_id"].astype(str)
    )
    if scores_long["unique_target"].nunique() < 2 or scores_long["judge_model"].nunique() < 2:
        icc_unique[dim] = np.nan
        continue
    try:
        icc_res = pg.intraclass_corr(
            data=scores_long,
            targets="unique_target",
            raters="judge_model",
            ratings="scores",
            nan_policy="omit",   # <-- fix: ignore missing/unbalanced rows for ICC
        )
        icc2k_row = icc_res[icc_res["Type"] == "ICC2k"]
        icc_unique[dim] = icc2k_row["ICC"].iloc[0] if not icc2k_row.empty else np.nan
    except ValueError as e:
        print(f"ICC calculation failed for dimension '{dim}': {e}")
        icc_unique[dim] = np.nan

icc_unique_series = pd.Series(icc_unique, name="icc_unique_target")
print("ICC with unique targets per dimension:")
print(icc_unique_series)

# Updated summary table with corrected ICC
if 'dim_summary' in globals():
    dim_summary_corrected = dim_summary.drop(columns=["icc"], errors="ignore").merge(
        icc_unique_series.reset_index().rename(columns={"index": "dimension"}),
        on="dimension",
        how="left",
    )
    dim_summary_corrected = dim_summary_corrected.rename(columns={"mean_std": "mean_std", "icc_unique_target": "icc"})
    print("\nCorrected dimension summary (with unique-target ICC):")
    display(dim_summary_corrected)


ICC with unique targets per dimension:
initiative_timing                   0.796611
interaction_coherence               0.840964
intent_alignment_drift              0.800839
commitment_consistency              0.813426
interaction_preference_alignment    0.794583
interaction_efficiency              0.845706
user_cognitive_load_trajectory      0.804157
overall_user_experience             0.804982
Name: icc_unique_target, dtype: float64

Corrected dimension summary (with unique-target ICC):


,dimension,mean_score,mean_std,icc
0,initiative_timing,3.787947,0.730622,0.796611
1,interaction_coherence,3.507838,0.630660,0.840964
2,intent_alignment_drift,4.412171,0.523927,0.800839
3,commitment_consistency,4.259213,0.571908,0.813426
4,interaction_preference_alignment,3.588361,0.847106,0.794583
5,interaction_efficiency,2.892108,0.607379,0.845706
6,user_cognitive_load_trajectory,3.346662,0.755041,0.804157
7,overall_user_experience,3.550056,0.737421,0.804982


In [36]:
# Compute ICC(2,1) per dimension (two-way random effects, single rater)
icc_21 = {}
for dim in DIMENSIONS:
    scores_long = all_scores_long[["tested_model", "condition", "persona", "test_id", "judge_model", dim]].copy()
    scores_long = scores_long.rename(columns={dim: "scores"})
    scores_long = scores_long.dropna(subset=["scores"])
    scores_long["unique_target"] = (
        scores_long["tested_model"]
        + "|" + scores_long["condition"]
        + "|" + scores_long["persona"]
        + "|" + scores_long["test_id"].astype(str)
    )
    if scores_long["unique_target"].nunique() < 2 or scores_long["judge_model"].nunique() < 2:
        icc_21[dim] = np.nan
        continue
    try:
        icc_res = pg.intraclass_corr(
            data=scores_long,
            targets="unique_target",
            raters="judge_model",
            ratings="scores",
            nan_policy="omit",
        )
        icc21_row = icc_res[icc_res["Type"] == "ICC2"]
        icc_21[dim] = icc21_row["ICC"].iloc[0] if not icc21_row.empty else np.nan
    except ValueError as e:
        print(f"ICC(2,1) calculation failed for dimension '{dim}': {e}")
        icc_21[dim] = np.nan

icc_21_series = pd.Series(icc_21, name="icc_2_1")
print("ICC(2,1) (single rater, two-way random effects) per dimension:")
print(icc_21_series)


ICC(2,1)（单评分者，双向随机效应）每个维度：
initiative_timing                   0.494739
interaction_coherence               0.569332
intent_alignment_drift              0.501313
commitment_consistency              0.521520
interaction_preference_alignment    0.491620
interaction_efficiency              0.578110
user_cognitive_load_trajectory      0.506546
overall_user_experience             0.507858
Name: icc_2_1, dtype: float64


In [37]:
# Create a new DataFrame with each dimension, ICC(2,1), ICC(2,k)
icc_comparison_df = pd.DataFrame({
    "dimension": DIMENSIONS,
    "icc_2_1": [icc_21_series.get(dim, np.nan) for dim in DIMENSIONS],
    "icc_2_k": [icc_unique_series.get(dim, np.nan) for dim in DIMENSIONS],
})
print("ICC(2,1) vs ICC(2,k) table:")
display(icc_comparison_df)


ICC(2,1) 与 ICC(2,k) 对比表：


,dimension,icc_2_1,icc_2_k
0,initiative_timing,0.494739,0.796611
1,interaction_coherence,0.569332,0.840964
2,intent_alignment_drift,0.501313,0.800839
3,commitment_consistency,0.521520,0.813426
4,interaction_preference_alignment,0.491620,0.794583
5,interaction_efficiency,0.578110,0.845706
6,user_cognitive_load_trajectory,0.506546,0.804157
7,overall_user_experience,0.507858,0.804982


## Cronbach's Alpha (UX scale internal consistency)
Compute per-sample judge-averaged scores across the 8 UX dimensions and estimate internal consistency using `pingouin.cronbach_alpha`.

In [38]:
# Average across judges per sample (unique target) for all 8 dimensions
# unique key ensures same dialogue instance is kept separate across model/condition/persona
avg_by_sample = (
    all_scores_long
    .groupby(["tested_model", "condition", "persona", "test_id"])[DIMENSIONS]
    .mean()
    .reset_index()
)

# Dataframe of averaged dimension scores only
df_avg = avg_by_sample[DIMENSIONS].copy()

# Cronbach's alpha across the 8 dimensions (treating dimensions as items)
alpha, ci = pg.cronbach_alpha(df_avg)
print(f"Cronbach's alpha for 8 UX dimensions (mean across judges per sample): {alpha:.4f}")
print(f"95% CI: {ci}")

# Show head of averaged scores for reference
df_avg.head()


Cronbach's alpha for 8 UX dimensions (mean across judges per sample): 0.9425
95% CI: [0.939 0.946]


,initiative_timing,interaction_coherence,intent_alignment_drift,commitment_consistency,interaction_preference_alignment,interaction_efficiency,user_cognitive_load_trajectory,overall_user_experience
0,2.25,3.75,3.75,3.50,2.00,2.00,2.25,2.75
1,3.25,3.25,5.00,5.00,3.00,3.25,3.50,3.25
2,3.50,4.75,4.75,4.75,2.50,3.25,3.00,3.00
3,3.00,3.25,4.25,4.25,3.75,2.75,3.00,3.25
4,2.25,2.00,2.75,2.50,2.75,2.00,2.00,2.50


## Two-way ANOVA (tested_model × judge_model)
For each UX dimension, run two-way ANOVA with score as DV and `tested_model` / `judge_model` as factors. Report Sum of Squares and partial eta-squared (np2).

In [39]:
anova_rows = []
for dim in DIMENSIONS:
    df_dim = all_scores_long[["tested_model", "judge_model", dim]].dropna()
    if df_dim.empty:
        continue
    res = pg.anova(data=df_dim, dv=dim, between=["tested_model", "judge_model"], detailed=True)
    res["dimension"] = dim
    anova_rows.append(res)

if anova_rows:
    anova_all = pd.concat(anova_rows, ignore_index=True)
    anova_summary = anova_all[["dimension", "Source", "SS", "np2", "p-unc"]]
    print("Two-way ANOVA summary (per dimension):")
    display(anova_summary)
else:
    print("No ANOVA results (empty data).")


Two-way ANOVA summary (per dimension):


,dimension,Source,SS,np2,p-unc
0,initiative_timing,tested_model,297.285101,0.024477,2.980780e-47
1,initiative_timing,judge_model,478.088867,0.038786,1.556494e-75
2,initiative_timing,tested_model * judge_model,30.589150,0.002575,6.624677e-03
3,initiative_timing,Residual,11848.146662,NaN,NaN
4,interaction_coherence,tested_model,687.171200,0.065109,7.672241e-129
...,...,...,...,...,...
27,user_cognitive_load_trajectory,Residual,11616.526569,NaN,NaN
28,overall_user_experience,tested_model,206.045934,0.016822,2.365562e-32
29,overall_user_experience,judge_model,1019.445475,0.078047,1.227762e-155
30,overall_user_experience,tested_model * judge_model,28.957582,0.002399,1.149120e-02


In [45]:
anova_summary.iloc[0:20]

,dimension,Source,SS,np2,p-unc
0,initiative_timing,tested_model,297.285101,0.024477,2.980780e-47
1,initiative_timing,judge_model,478.088867,0.038786,1.556494e-75
2,initiative_timing,tested_model * judge_model,30.589150,0.002575,6.624677e-03
3,initiative_timing,Residual,11848.146662,NaN,NaN
4,interaction_coherence,tested_model,687.171200,0.065109,7.672241e-129
5,interaction_coherence,judge_model,570.006065,0.054614,2.007884e-107
6,interaction_coherence,tested_model * judge_model,18.326673,0.001854,5.840981e-02
7,interaction_coherence,Residual,9866.932795,NaN,NaN
8,intent_alignment_drift,tested_model,450.762623,0.058032,4.916978e-114
9,intent_alignment_drift,judge_model,536.088211,0.068267,6.177422e-135


In [44]:
anova_summary.iloc[20:]

,dimension,Source,SS,np2,p-unc
20,interaction_efficiency,tested_model,514.331347,0.048571,3.097995e-95
21,interaction_efficiency,judge_model,579.625299,0.054402,4.976287e-107
22,interaction_efficiency,tested_model * judge_model,38.742309,0.003831,8.947563e-05
23,interaction_efficiency,Residual,10074.843106,NaN,NaN
24,user_cognitive_load_trajectory,tested_model,404.894569,0.033681,2.483469e-65
25,user_cognitive_load_trajectory,judge_model,1483.673546,0.113256,5.618214e-230
26,user_cognitive_load_trajectory,tested_model * judge_model,55.658451,0.004768,2.950086e-06
27,user_cognitive_load_trajectory,Residual,11616.526569,NaN,NaN
28,overall_user_experience,tested_model,206.045934,0.016822,2.365562e-32
29,overall_user_experience,judge_model,1019.445475,0.078047,1.227762e-155
